In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    ThermalizationConfig,
    run_thermalization,
)

# ==================================================
# User inputs
# ==================================================

dts = [
    0.001,
    0.002,
    0.0025,
    0.005,
    0.01,
]

# Use one seed or multiple seeds.
seeds = []
for i in range(0,10):
    seeds.append(i)

kT = 0.9
density = 0.7
n_fcc_cells = 30

# Every simulation will evolve for approximately this
# amount of LJ time.
target_lj_time = 100.0

# Each plotted point is the mean of the final 100
# logged samples. Its error bar is the standard error
# of that mean: sample std / sqrt(number of samples).
summary_points = 100

# Generate approximately this many logged points per
# simulation. Keeping this constant makes the final
# 100 samples span approximately the same LJ-time
# interval for every dt.
target_logged_points = 200

# Set True to use a logarithmic dt axis.
use_log_dt_axis = False

# Small visual offset so points from different seeds
# at the same dt do not cover one another.
seed_jitter_fraction = 0.018

# ==================================================
# Setup
# ==================================================

paths = ProjectPaths()
database = SQLiteRunDatabase(paths.database)
database.initialize()

dts = np.asarray(dts, dtype=float)
seeds = [int(seed) for seed in seeds]

if np.any(dts <= 0):
    raise ValueError("Every dt must be positive")

if target_lj_time <= 0:
    raise ValueError("target_lj_time must be positive")

if target_logged_points < summary_points:
    raise ValueError(
        "target_logged_points must be at least summary_points"
    )

results = []

# ==================================================
# Run every dt × seed combination
# ==================================================

for dt in dts:
    # This keeps Nsteps * dt approximately constant.
    nsteps = int(round(target_lj_time / dt))

    if nsteps < target_logged_points:
        raise ValueError(
            f"dt={dt:g} gives only {nsteps:,} steps. "
            f"Increase target_lj_time or reduce "
            f"target_logged_points."
        )

    # Floor division ensures at least approximately
    # target_logged_points evolved log samples.
    log_period = max(
        1,
        nsteps // target_logged_points,
    )

    actual_lj_time = nsteps * dt
    approximate_log_lj_interval = log_period * dt

    for seed in seeds:
        print(
            f"\ndt={dt:g}, seed={seed}, "
            f"Nsteps={nsteps:,}, "
            f"LJ time={actual_lj_time:.6g}, "
            f"log period={log_period:,} steps"
        )

        config = ThermalizationConfig(
            n_fcc_cells=n_fcc_cells,
            target_rho=density,
            nsteps=nsteps,
            kT=kT,
            seed=seed,
            dt=float(dt),
            log_period=log_period,
            summary_num_samples=summary_points,
            pe_drop_n_last=summary_points,
            notes=(
                "dt comparison; "
                f"dt={dt:g}; "
                f"seed={seed}; "
                f"target_lj_time={target_lj_time:g}"
            ),
        )

        run_result = run_thermalization(
            config,
            project_paths=paths,
            database=database,
        )

        run_id = run_result["run_id"]
        status = run_result["status"]

        if status != "Complete":
            raise RuntimeError(
                f"Run {run_id} for dt={dt:g}, seed={seed} "
                f"has Status={status!r}, not 'Complete'."
            )

        summary = database.get_thermalization(run_id)

        if summary is None:
            raise RuntimeError(
                f"Run {run_id} has no Thermalization result row"
            )

        num_samples = int(summary["Summary_Num_Samples"])

        if num_samples != summary_points:
            raise RuntimeError(
                f"Run {run_id} used {num_samples} summary "
                f"samples instead of {summary_points}."
            )

        pressure_std = float(summary["Pressure_Std"])
        pe_per_particle_std = float(
            summary["PE_Per_Particle_Std"]
        )

        # Standard error of each 100-point mean.
        pressure_sem = pressure_std / np.sqrt(num_samples)
        pe_per_particle_sem = (
            pe_per_particle_std / np.sqrt(num_samples)
        )

        results.append(
            {
                "Run_ID": run_id,
                "dt": float(dt),
                "seed": seed,
                "Nsteps": nsteps,
                "actual_lj_time": actual_lj_time,
                "log_period": log_period,
                "log_lj_interval": (
                    approximate_log_lj_interval
                ),
                "summary_samples": num_samples,
                "pressure_mean": float(
                    summary["Pressure_Mean"]
                ),
                "pressure_std": pressure_std,
                "pressure_sem": pressure_sem,
                "pe_per_particle_mean": float(
                    summary["PE_Per_Particle_Mean"]
                ),
                "pe_per_particle_std": pe_per_particle_std,
                "pe_per_particle_sem": pe_per_particle_sem,
                "reused_existing_run": bool(
                    run_result.get("skipped", False)
                ),
            }
        )

# Results table is retained for later analysis.
dt_results = (
    pd.DataFrame(results)
    .sort_values(["seed", "dt"])
    .reset_index(drop=True)
)

display(dt_results)

# Add/update the Master-table note for every dt-test run.
dt_test_note = "These runs are for the dt test."

for run_id in dt_results["Run_ID"].unique():
    database.update_master(
        run_id,
        Notes=dt_test_note,
    )

print(
    f"Updated Notes for "
    f"{dt_results['Run_ID'].nunique()} dt-test runs."
)

# ==================================================
# Plot mean ± standard error for each seed
# ==================================================

fig, (ax_pressure, ax_pe) = plt.subplots(
    1,
    2,
    figsize=(15, 6),
)

colors = plt.cm.tab10(
    np.linspace(0, 1, len(seeds))
)

if len(seeds) == 1:
    jitter_offsets = np.array([0.0])
else:
    jitter_offsets = np.linspace(
        -seed_jitter_fraction,
        seed_jitter_fraction,
        len(seeds),
    )

for color, seed, jitter in zip(
    colors,
    seeds,
    jitter_offsets,
):
    seed_results = (
        dt_results[dt_results["seed"] == seed]
        .sort_values("dt")
    )

    true_dt = seed_results["dt"].to_numpy()

    # Jitter changes only the displayed x-coordinate,
    # not the simulated dt.
    plotted_dt = true_dt * (1.0 + jitter)

    # Pressure: final-100-point mean ± SEM.
    ax_pressure.errorbar(
        plotted_dt,
        seed_results["pressure_mean"],
        yerr=seed_results["pressure_sem"],
        color=color,
        marker="o",
        markersize=6,
        linewidth=1.5,
        capsize=4,
        alpha=0.85,
        label=f"Seed {seed}",
    )

    # PE/N: final-100-point mean ± SEM.
    ax_pe.errorbar(
        plotted_dt,
        seed_results["pe_per_particle_mean"],
        yerr=seed_results["pe_per_particle_sem"],
        color=color,
        marker="o",
        markersize=6,
        linewidth=1.5,
        capsize=4,
        alpha=0.85,
        label=f"Seed {seed}",
    )

# ==================================================
# Formatting
# ==================================================

ax_pressure.set_title(
    f"Pressure: final {summary_points}-point mean ± SEM"
)
ax_pressure.set_xlabel("Integration timestep, dt")
ax_pressure.set_ylabel("Pressure")

ax_pe.set_title(
    f"PE/N: final {summary_points}-point mean ± SEM"
)
ax_pe.set_xlabel("Integration timestep, dt")
ax_pe.set_ylabel("Potential energy per particle, PE/N")

for ax in (ax_pressure, ax_pe):
    if use_log_dt_axis:
        ax.set_xscale("log")

    ax.set_xticks(np.sort(np.unique(dts)))
    ax.set_xticklabels(
        [f"{dt:g}" for dt in np.sort(np.unique(dts))]
    )
    ax.grid(alpha=0.3)
    ax.legend(title="Random seed")

fig.suptitle(
    (
        f"dt convergence at kT={kT:g}, ρ={density:g}, "
        f"Ncells={n_fcc_cells}, "
        f"LJ time ≈ {target_lj_time:g}"
    ),
    fontsize=14,
)

plt.tight_layout()
plt.show()


dt=0.001, seed=0, Nsteps=100,000, LJ time=100, log period=500 steps
